In [ ]:
from utils.py_eddy_tracker.dataset.grid import RegularGridDataset
from utils.py_eddy_tracker.eddy_feature import Amplitude, Contours

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from datetime import datetime
from pathlib import Path
from typing import Optional


def get_axes(title: str, xlim: tuple[float, float], ylim: tuple[float, float]) -> plt.Axes:
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_axes((0.03, 0.03, 0.9, 0.94))
    ax.set_xlim(xlim[0], xlim[1])
    ax.set_ylim(ylim[0], ylim[1])
    ax.set_aspect("equal")
    ax.set_title(title, weight="bold")
    return ax


def update_axes(ax: plt.Axes, mappable: Optional[ScalarMappable] = None) -> None:
    ax.grid()
    if mappable is not None:
        plt.colorbar(mappable, cax=ax.figure.add_axes((0.94, 0.05, 0.01, 0.9)))

### Display the SWOT L4 Data

In [ ]:
fp = "/Users/jerry/school/research/eddy-tracking/data/gulf_stream_pigment_influencers_20241001_20251231/bronze/swot_l4/dt_global_allsat_phy_l4_20230328_20250112.nc"
ds = xr.open_dataset(fp)
lon_range = (ds["longitude"].min().item(), ds["longitude"].max().item())
lat_range = (ds["latitude"].min().item(), ds["latitude"].max().item())
print(lon_range)
print(lat_range)

In [ ]:
xlim = (-81, -56)
ylim = (29, 44)
g = RegularGridDataset(
    filename=fp,
    x_name='longitude',
    y_name='latitude',
    indexs={
        'time': 0
    }
)
ax = get_axes('SWOT L4 Gulf Stream, ADT', xlim, ylim)
m: ScalarMappable = g.display(ax, 'adt', vmin=-1, vmax=1, cmap='RdBu_r')

In [ ]:
ds["longitude"]

In [ ]:
# Geostrophy: surface current is proportional to the sea surface height slope, so the current core sits where ADT changes fastest.
lon = ds["longitude"].to_numpy() # 1D arrays
lat = ds["latitude"].to_numpy()
lon_min, lon_max = lon_range
lat_min, lat_max = lat_range

ugos = ds["ugos"].isel(time=0, drop=True).to_numpy() # (time, n_lat, n_lon) -> (n_lat, n_lon); NaN over land
vgos = ds["vgos"].isel(time=0, drop=True).to_numpy()
speed = np.hypot(ugos, vgos)

import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
fig, ax = plt.subplots(figsize=(12, 8))
mesh = plt.pcolormesh(ds["longitude"], ds["latitude"], speed)
fig.colorbar(mesh, ax=ax, label="speed (m/s)")

class GulfStreamCenterline:
    def __init__(self, lon: np.ndarray, lat: np.ndarray) -> None:
        self.lon = lon # both 1D arrays of the same length representing pixels of Gulf Stream
        self.lat = lat

    @classmethod
    def from_speed_field(
        cls,
        speed: np.ndarray,
        lon: np.ndarray,
        lat: np.ndarray,
        speed_threshold_percentile: int = 70,
    ) -> "GulfStreamCenterline":
        """
        Find the Gulf Stream core by following the fastest-flowing water.

        Starts at the fastest cell (the origin) and moves outward one column at a time, each step only looking within window_deg latitude of the previous column's core.

        Args:
            speed: 2D speed grid, shape (n_lat, n_lon); speed[i, j] is at lat[i], lon[j].
            lon: 1D longitudes, shape (n_lon,).
            lat: 1D latitudes, shape (n_lat,).
            speed_threshold_percentile: percentile on a 0-100 scale; columns whose best speed falls below this cutoff are dropped.

        Returns:
            GulfStreamCenterline whose lon and lat are matching 1D arrays of the core points (no-current columns dropped), ordered west to east.
        """
        threshold = np.nanpercentile(speed, speed_threshold_percentile)
        window_deg = 1.0 # how far the core may move in latitude between adjacent columns

        track = np.full(lon.shape, np.nan) # core latitude per longitude column, NaN where no current
        origin_flat_idx = np.nanargmax(speed)
        origin_i, origin_j = np.unravel_index(origin_flat_idx, speed.shape) # flat index into (n_lat, n_lon) -> (row, col)
        track[origin_j] = lat[origin_i]

        def core_in_window(j: int, prev_lat: float) -> float:
            column = np.where(np.abs(lat - prev_lat) <= window_deg, speed[:, j], np.nan)
            if np.all(np.isnan(column)):
                return np.nan
            i = np.nanargmax(column)
            return lat[i] if column[i] >= threshold else np.nan

        prev_lat = lat[origin_i]
        for j in range(origin_j + 1, len(lon)): # go east from the origin
            track[j] = core_in_window(j, prev_lat)
            if np.isfinite(track[j]):
                prev_lat = track[j]

        prev_lat = lat[origin_i]
        for j in range(origin_j - 1, -1, -1): # go west from the origin
            track[j] = core_in_window(j, prev_lat)
            if np.isfinite(track[j]):
                prev_lat = track[j]

        ok = np.isfinite(track)
        return cls(lon[ok], track[ok])

    @classmethod
    def from_streamline_field(
        cls,
        ugos: np.ndarray,
        vgos: np.ndarray,
        lon: np.ndarray,
        lat: np.ndarray,
        speed_threshold_percentile: int = 70,
    ) -> "GulfStreamCenterline":
        """
        Trace the Gulf Stream core by following the flow direction (a streamline).

        Drops a point at the fastest cell (the origin) and steps along the local velocity direction, so the path can curve, loop, and double back with the stream instead of assuming one latitude per longitude.
        To get past small breaks in the jet it coasts through up to max_gap_steps slow cells before giving up, then trims any weak tail left dangling at the end.
        Stops at the grid edge, on land, or where the path curls back onto itself (a ring).
        Needs the velocity components, not just speed, since it follows direction.

        Args:
            ugos: 2D eastward velocity, shape (n_lat, n_lon).
            vgos: 2D northward velocity, shape (n_lat, n_lon).
            lon: 1D longitudes, shape (n_lon,).
            lat: 1D latitudes, shape (n_lat,).
            speed_threshold_percentile: percentile on a 0-100 scale; cells slower than this cutoff count as gaps to coast over.

        Returns:
            GulfStreamCenterline whose lon and lat are the traced path in order (not sorted by longitude, since the path may double back).
        """
        speed = np.hypot(ugos, vgos)
        threshold = np.nanpercentile(speed, speed_threshold_percentile)
        u_at = RegularGridInterpolator((lat, lon), ugos, bounds_error=False, fill_value=np.nan)
        v_at = RegularGridInterpolator((lat, lon), vgos, bounds_error=False, fill_value=np.nan)

        origin_i, origin_j = np.unravel_index(np.nanargmax(speed), speed.shape)
        origin = np.array([lat[origin_i], lon[origin_j]]) # (lat, lon)

        def trace(direction: int) -> list:
            # direction +1 follows the flow downstream, -1 traces upstream
            step_km = 5.0
            max_gap_steps = 10 # slow cells to coast through before giving up (~50 km)
            point = origin.copy()
            path = []
            gap = 0
            last_strong = -1 # last point on the jet, used to trim a dangling slow tail
            for _ in range(2000):
                u = float(u_at([point])[0])
                v = float(v_at([point])[0])
                spd = np.hypot(u, v)
                if not np.isfinite(spd) or spd == 0:
                    break
                path.append(point.copy())
                if spd >= threshold:
                    gap = 0
                    last_strong = len(path) - 1
                else:
                    gap += 1
                    if gap > max_gap_steps:
                        break
                # step step_km along the unit flow direction, converted to degrees
                point = point + np.array([
                    step_km / 111.0 * direction * v / spd,
                    step_km / (111.0 * np.cos(np.radians(point[0]))) * direction * u / spd,
                ])
                if not (lat.min() <= point[0] <= lat.max() and lon.min() <= point[1] <= lon.max()):
                    break
                if len(path) > 40: # stop if we curl back onto an earlier part (ring)
                    # list of (2,) points -> (n_earlier, 2) with columns (lat, lon)
                    earlier = np.array(path[:-30])
                    if np.hypot(earlier[:, 0] - point[0], earlier[:, 1] - point[1]).min() < step_km / 111.0:
                        break
            return path[:last_strong + 1] # drop any dangling slow tail

        upstream = trace(-1)
        downstream = trace(+1)
        path = np.array(upstream[::-1] + downstream[1:]) # two lists of (2,) points -> (n_points, 2); the origin is in both traces, so keep it once
        return cls(path[:, 1], path[:, 0]) # (n_points, 2) rows are (lat, lon), so pass lon then lat

    def is_pixel_north(self, pixel: tuple[float, float]) -> bool:
        """Returns whether a pixel (lon, lat) is north of the Gulf Stream. The Gulf Stream line is determined through linear interpolation of centerline pixels."""
        pixel_lon, pixel_lat = pixel
        stream_lat = np.interp(pixel_lon, self.lon, self.lat)
        return bool(pixel_lat > stream_lat) # cast because the numpy comparison returns np.bool_


gulf_stream = GulfStreamCenterline.from_speed_field(speed, lon, lat, speed_threshold_percentile=70)
ax.plot(gulf_stream.lon, gulf_stream.lat, color="black", label="Gulf Stream by Max Speed")

gulf_stream_streamline = GulfStreamCenterline.from_streamline_field(ugos, vgos, lon, lat, speed_threshold_percentile=70)
ax.plot(gulf_stream_streamline.lon, gulf_stream_streamline.lat, color="red", label="Gulf Stream by Streamline")
ax.legend()

### Preprocessing

ADT after a 600 km Bessel high-pass filter to remove the basin-scale signal, isolating mesoscale eddy features. Gulf Stream mean dynamic topography overlaid in black as a geographic reference.

In [ ]:
g.bessel_high_filter('adt', 600)
ax = get_axes('Gulf Stream, ADT (m) filtered (600 km), 2023-03-28', xlim, ylim)
m = g.display(ax, 'adt', vmin=-0.5, vmax=0.5, cmap='RdBu_r')
ax.plot(gulf_stream.lon, gulf_stream.lat, color='k')
update_axes(ax, m)

### Identification

Eddy identification results overlaid on filtered ADT. Closed contours detected by py-eddy-tracker (step = 0.004 m, shape error ≤ 50%) are shown for both cyclonic and anticyclonic eddies on 2023-03-28.

In [ ]:
date = datetime(2023, 3, 28)
anticyclones, cyclones = g.eddy_identification(
    grid_height='adt',
    uname='ugos',
    vname='vgos',
    date=date,
    step=0.004,
    shape_error=50
)
ax = get_axes('Gulf Stream, ADT (m) filtered (600 km) closed eddy contours, 2023-03-28', xlim, ylim)
sm = ScalarMappable(norm=Normalize(vmin=-0.4, vmax=0.4), cmap='RdYlBu_r')
g.contours.display(ax, lw=0.25, only_used=True) #type:ignore
ax.plot(gulf_stream.lon, gulf_stream.lat, color='k')
update_axes(ax, sm)

### EddyNet Comparison

In [ ]:
import torch

In [ ]:
model: torch.nn.Module = torch.hub.load("edwinytgoh/eddynet", "eddynet", pretrained=True, num_classes=3) #type:ignore
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

In [ ]:
# (1, n_lat, n_lon) -> (n_lat, n_lon)
adt = ds.data_vars['adt'].to_numpy().squeeze()
adt = np.nan_to_num(adt, nan=0.0) # land -> 0 so the network gets finite input

# EddyNet pools by 2 three times, so its input dims must be divisible by 8; pad up, then crop the mask back
h, w = adt.shape
h_pad, w_pad = ((h + 7) // 8) * 8, ((w + 7) // 8) * 8
padded = np.zeros((h_pad, w_pad), dtype=adt.dtype)
padded[:h, :w] = adt

# (h_pad, w_pad) -> (1, 1, h_pad, w_pad)
adt_tensor = torch.from_numpy(padded).reshape(1, 1, h_pad, w_pad).float().to(device)
logits = model(adt_tensor)
# (1, 3, h_pad, w_pad) -> (1, h_pad, w_pad) -> (h_pad, w_pad) -> (h, w)
eddy_mask = torch.argmax(logits, dim=1).squeeze().cpu().numpy()[:h, :w]

Comparison of filtered ADT field (top) and EddyNet deep-learning segmentation (bottom). EddyNet classifies each grid cell as background (gray), cyclonic (blue), or anticyclonic (red) using a pretrained CNN.

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm

fig, ax = plt.subplots(2, 1, figsize=(10, 10))

# PET stores a grid as (n_lon, n_lat): (n_lon, n_lat) -> (n_lat, n_lon), then flip lat so imshow draws north at the top
im = ax[0].imshow(g.grid('adt').T[::-1, ...], extent=[lon_min, lon_max, lat_min, lat_max], aspect='equal', vmin=-0.5, vmax=0.5, cmap='RdBu_r')
ax[0].set_title('ADT (m) filtered (600 km), 2023-03-28')
plt.colorbar(im, ax=ax[0])

im2 = ax[1].imshow(g.grid('adt').T[::-1, ...], extent=[lon_min, lon_max, lat_min, lat_max], aspect='equal', vmin=-0.5, vmax=0.5, cmap='RdBu_r')
cmap = ListedColormap(['none', 'red', 'blue'])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)
ax[1].imshow(eddy_mask[::-1, ...], extent=[lon_min, lon_max, lat_min, lat_max], aspect='equal', cmap=cmap, norm=norm, alpha=0.5)
ax[1].set_title('EddyNet Detected Eddies on 2023-03-28')
plt.colorbar(im2, ax=ax[1])
plt.tight_layout()

### PET Comparison

Filtered ADT (top) with py-eddy-tracker contour overlay (bottom). Dashed contours mark the effective radius (outer boundary); solid contours mark the speed radius (maximum rotational velocity). Red = anticyclonic, blue = cyclonic.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 10))

# PET stores a grid as (n_lon, n_lat): (n_lon, n_lat) -> (n_lat, n_lon), then flip lat so imshow draws north at the top
im = ax[0].imshow(g.grid('adt').T[::-1, ...], extent=[lon_min, lon_max, lat_min, lat_max], aspect='equal', vmin=-0.5, vmax=0.5, cmap='RdBu_r')
ax[0].set_title('ADT (m) filtered (600 km), 2023-03-28')
plt.colorbar(im, ax=ax[0])

im2 = ax[1].imshow(g.grid('adt').T[::-1, ...], extent=[lon_min, lon_max, lat_min, lat_max], aspect='equal', vmin=-0.5, vmax=0.5, cmap='RdBu_r')
anticyclones.display(ax[1], color='r', linewidth=2, label='anticyclonic', ref=-180)
cyclones.display(ax[1], color='b', linewidth=2, label='cyclonic', ref=-180)
ax[1].set_xlim(lon_min, lon_max)
ax[1].set_ylim(lat_min, lat_max)
ax[1].legend()
ax[1].set_title('PET Detected Eddies on 2023-03-28')
plt.colorbar(im2, ax=ax[1])
plt.tight_layout()

In [ ]:
plt.close('all')

### Working with Identification File Outputs

Files like `Anticyclonic_20190223.nc` are daily snapshots of all eddies detected on a specific date.

**Dimensions:**
- `obs` - number of eddies detected that day (e.g., 3137 anticyclonic eddies on 2019-02-23)
- `NbSample` - number of points used to represent contours (fixed at 50)

**Core Variables (1D, shape: obs):**
- `longitude`, `latitude` - Eddy center coordinates
- `amplitude` - Height difference between center and edge (cm)
- `time` - Observation timestamp

**Effective Contour (outer boundary):**
- `effective_radius` - Radius of outer contour (km)
- `effective_area` - Area enclosed by outer contour
- `effective_contour_height` - SSH value at this contour
- `effective_contour_shape_error` - Circularity measure (0 = perfect circle)
- `effective_contour_longitude/latitude` - Contour coordinates (shape: obs × NbSample)

**Speed Contour (max rotational velocity):**
- `speed_radius` - Radius of max speed contour (km)
- `speed_area` - Area enclosed
- `speed_average` - Mean rotational speed at this contour
- `speed_contour_height` - SSH value at this contour
- `speed_contour_shape_error` - Circularity measure
- `speed_contour_longitude/latitude` - Contour coordinates (shape: obs × NbSample)

**Profile Data:**
- `uavg_profile` - Azimuthally-averaged velocity profile (shape: obs × NbSample)

In [ ]:
from utils.py_eddy_tracker import data
from utils.py_eddy_tracker.observations.observation import EddiesObservations

In [ ]:
a = EddiesObservations.load_file(data.get_demo_path("Anticyclonic_20190223.nc"))
c = EddiesObservations.load_file(data.get_demo_path("Cyclonic_20190223.nc"))